# 1. Ransomware Resiliency — Architecture Design

SC-100 doesn't ask *"what is ransomware?"* — it asks:

> *"Design a resiliency strategy for Contoso that protects their critical assets and ensures business continuity."*

This notebook walks you through that design, **showing a bad approach first and then iterating to the recommended one**. Each step has runnable code you can tweak.

## The 3-phase mental model

```
┌─────────────────────────────────────────────────────────────────┐
│  PREPARE (before)   identify assets · protect identities ·      │
│                     immutable backups · tested restore          │
├─────────────────────────────────────────────────────────────────┤
│  DETECT (during)    Defender XDR + Sentinel correlation ·       │
│                     automatic attack disruption                 │
├─────────────────────────────────────────────────────────────────┤
│  RECOVER (after)    BCDR playbook · restore from immutable ·    │
│                     rebuild identities · rotate secrets         │
└─────────────────────────────────────────────────────────────────┘
```

## Setup — run this once

This lab uses only the Python standard library (no extra dependencies).

1. Open a terminal in `security-certs/sc-100/01-best-practices/`.
2. Run `uv sync` to create the lab's `.venv`.
3. In VS Code, click the kernel picker (top-right of this notebook) and select **.venv (Python)**.
4. If the kernel doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Then run the cell below to confirm Python is working.

In [ ]:
import sys, platform
print('Python :', sys.version.split()[0])
print('Kernel :', sys.executable)
print('OS     :', platform.system(), platform.release())
print('Ready to design secure architectures ✅')

## The scenario we'll design for

We'll design a 90-day plan for **Contoso Financial Services**. The scenario is deliberately realistic — messy current state, hard deadline, board pressure — because that's how the exam frames it.

In [ ]:
SCENARIO = '''
COMPANY: Contoso Financial Services  (5,000 employees)
INFRASTRUCTURE:
  - Azure: 3 subscriptions (prod, staging, dev)
  - On-premises: Active Directory, legacy financial apps
  - Microsoft 365: Exchange Online, SharePoint, Teams
  - 200 VMs across Azure and on-prem
  - Azure SQL databases with customer financial data
  - Storage accounts with regulatory documents

CURRENT STATE (the problems):
  - Azure Backup configured but NEVER tested
  - No immutable backups
  - Global admin accounts don't use PIM
  - Defender for Endpoint on 60% of devices
  - No Sentinel deployment
  - Conditional Access: MFA for admins only
  - Network: flat, no segmentation

REQUIREMENT: Board mandates ransomware readiness within 90 days,
triggered by a peer firm taking a $50M ransomware hit.
'''
print(SCENARIO)

## Step 1 — Know what you're protecting (RTO / RPO)

Before any tech decision, an architect classifies assets by **business impact** and assigns:

- **RTO** (Recovery Time Objective) — *how fast must it be back?*
- **RPO** (Recovery Point Objective) — *how much data loss is acceptable?*

Without these numbers you can't choose between backup, replication, or geo-redundancy.

In [ ]:
CRITICAL_ASSETS = [
    {'asset': 'Customer financial DB (Azure SQL)',   'impact': 'Critical', 'rto_min': 60,   'rpo_min': 15},
    {'asset': 'Entra ID / AD DS',                    'impact': 'Critical', 'rto_min': 120,  'rpo_min': 0},
    {'asset': 'Microsoft 365 (Exchange, SharePoint)','impact': 'High',     'rto_min': 240,  'rpo_min': 60},
    {'asset': 'Regulatory document storage',         'impact': 'High',     'rto_min': 480,  'rpo_min': 1440},
    {'asset': 'Legacy financial apps (on-prem VMs)', 'impact': 'High',     'rto_min': 240,  'rpo_min': 60},
    {'asset': 'Development environment',             'impact': 'Medium',   'rto_min': 1440, 'rpo_min': 1440},
]

def fmt_min(m):
    if m == 0: return '0 (no loss)'
    if m < 60: return f'{m} min'
    if m < 1440: return f'{m // 60} h'
    return f'{m // 1440} d'

print(f'{"Asset":<42} {"Impact":<10} {"RTO":<10} {"RPO":<10}')
print('-' * 75)
for a in CRITICAL_ASSETS:
    print(f'{a["asset"]:<42} {a["impact"]:<10} {fmt_min(a["rto_min"]):<10} {fmt_min(a["rpo_min"]):<10}')

## Step 2 — Backup strategy: bad → better → best

This is the heart of ransomware resiliency. Let's evolve a design.

In [ ]:
# ❌ BAD: 'We have Azure Backup enabled'
bad = {
    'backup_type':       'Azure Backup (default)',
    'immutable':         False,           # attacker with perms can delete
    'cross_region':      False,           # region outage = data loss
    'offline_copy':      False,           # all copies reachable from network
    'restore_tested':    False,           # backups may silently be corrupt
    'admin_protection':  'same admins as production',
}

# ⚠️ BETTER: 'We added GRS + soft delete'
better = {
    'backup_type':       'Azure Backup + GRS storage',
    'immutable':         False,
    'cross_region':      True,
    'offline_copy':      False,
    'restore_tested':    False,
    'admin_protection':  'separate backup operator role',
}

# ✅ BEST: the 3-2-1-1-0 rule Microsoft recommends for ransomware
best = {
    'backup_type':       'Azure Backup with Recovery Services Vault',
    'immutable':         True,            # WORM lock — even admins can't delete
    'cross_region':      True,            # paired region
    'offline_copy':      True,            # M365 Backup + long-term retention tier
    'restore_tested':    True,            # monthly full-restore drill
    'admin_protection':  'separate tenant / MFA / PIM for backup admins',
    'rule':              '3 copies, 2 media, 1 offsite, 1 immutable, 0 restore errors',
}

def score(plan):
    keys = ['immutable', 'cross_region', 'offline_copy', 'restore_tested']
    return sum(1 for k in keys if plan.get(k)) * 25

for label, plan in [('BAD   ', bad), ('BETTER', better), ('BEST  ', best)]:
    print(f'{label}  resilience score: {score(plan):>3}/100')

### Why "immutable" is the single most important property

Modern ransomware actors **specifically hunt for backups** before detonating — if they can delete or encrypt them, you have to pay. Immutable storage makes the backup un-deletable for a defined retention window, even by someone with admin credentials.

Exam trigger phrases → answer:

| Phrase in the question | Pick this |
|---|---|
| "protect backups from ransomware" | **Immutable vault** + resource locks |
| "recover critical DB within 1 hour" | **Active geo-replication / failover group** (not daily backup) |
| "limit blast radius of compromised admin" | **PIM** + Privileged Access Workstation |
| "validate the plan works" | **Restore drill** to an isolated environment |

## Step 3 — Privileged access: bad → best

Most ransomware incidents **start with a compromised identity**, not malware. Protecting admin accounts reduces blast radius more than any endpoint product can.

In [ ]:
identity_maturity = [
    ('❌ BAD',    'Permanent Global Admins, shared password, no MFA'),
    ('⚠️ OK',     'MFA enabled, but standing admin access'),
    ('🟡 GOOD',   'MFA + PIM just-in-time elevation'),
    ('✅ BEST',   'Phishing-resistant MFA + PIM + PAW + 2 break-glass accounts (FIDO2 key in a safe, excluded from CA)'),
]
for level, desc in identity_maturity:
    print(f'{level:<12} {desc}')

print()
print('PIM = Privileged Identity Management: admin gets role for 1h, with approval, logged.')
print('PAW = dedicated hardened workstation used ONLY for admin tasks — no email, no browsing.')
print('Break-glass = 2+ cloud-only (*.onmicrosoft.com) Global Admin accounts, PERMANENTLY active in PIM,')
print('              secured with phishing-resistant MFA (FIDO2 passkey or certificate-based auth) whose')
print('              key/credential lives in a physical safe, and EXCLUDED from Conditional Access policies')
print('              that block or restrict sign-in. Validate they still work every 90 days.')
print()
print('⚠️  Outdated advice you will still see: "break-glass accounts should have no MFA".')
print('    That is no longer Microsoft guidance. Mandatory MFA now applies to admin portals, and the current')
print('    guidance is a phishing-resistant method DIFFERENT from the one your normal admin accounts use.')
print('    The resilience you want comes from the CA exclusion, not from removing the second factor.')

## Step 4 — A runnable posture calculator

Here's a tiny scoring model that mirrors how Defender for Cloud's Secure Score works. It reads Contoso's current state, weights each control by its ransomware impact, and prints what to fix first.

In [ ]:
CONTROLS = [
    # (control, implemented?, weight, category)
    ('MFA for all users',              False, 20, 'Identity'),
    ('PIM for admin roles',            False, 15, 'Identity'),
    ('Block legacy authentication',    False, 10, 'Identity'),
    ('Immutable backup vault',         False, 20, 'Backup'),
    ('Tested restore drill',           False, 10, 'Backup'),
    ('Cross-region replication',       False,  5, 'Backup'),
    ('Defender for Endpoint everywhere', False, 10, 'Detect'),
    ('Sentinel SIEM deployed',         False,  5, 'Detect'),
    ('Network segmentation',           False,  5, 'Network'),
]

def posture_score(controls):
    earned = sum(w for _, ok, w, _ in controls if ok)
    total  = sum(w for _, _, w, _ in controls)
    return earned, total

earned, total = posture_score(CONTROLS)
print(f'Current score: {earned}/{total}  ({earned*100//total}%)')

print('\nTop fixes ranked by weight:')
for name, ok, w, cat in sorted(CONTROLS, key=lambda c: -c[2]):
    if not ok:
        print(f'  + {w:>2} pts  [{cat:<8}] {name}')

In [ ]:
# Now simulate implementing the top 5 — watch the score jump
fixed = [(n, True if i < 5 else ok, w, c) for i, (n, ok, w, c) in enumerate(sorted(CONTROLS, key=lambda c: -c[2]))]
e, t = posture_score(fixed)
print(f'After Phase 1 fixes: {e}/{t}  ({e*100//t}%)  — biggest gains come from identity + immutable backup.')

## Step 5 — The 90-day plan

Phasing matters on the exam. Microsoft's guidance: **identity → backup → detection → network → endpoint**. Quick wins go first because they reduce risk fastest and unlock later work.

In [ ]:
PHASES = [
    ('Weeks 1-2  (Quick wins)', [
        'Enable PIM for all admin roles',
        'MFA for ALL users (not just admins)',
        'Switch backup vaults to immutable',
        'Block legacy authentication',
        'Roll Defender for Endpoint to remaining 40%',
    ]),
    ('Weeks 3-6  (Core protection)', [
        'Deploy Microsoft Sentinel (central SIEM)',
        'Enable Defender for Cloud plans (Servers P2, SQL, Storage)',
        'Network segmentation: NSGs + Azure Firewall hub',
        'Azure SQL active geo-replication / failover groups → meet 1-h RTO',
        'Azure Site Recovery (ASR) replication for critical on-prem VMs',
        'Create 2 break-glass admin accounts (FIDO2, excluded from CA, permanently active in PIM)',
    ]),
    ('Weeks 7-12 (Mature & test)', [
        'Full BCDR restore drill (not just backup verification!)',
        'Ransomware incident-response playbook',
        'Tabletop exercise with leadership',
        'Conditional Access: risk-based + device compliance',
        'Sentinel analytics rules for ransomware TTPs',
        'AD tier model + GPO hardening',
    ]),
]

for phase, actions in PHASES:
    print(f'\n📅 {phase}')
    for a in actions:
        print(f'   ☐ {a}')

## Key takeaways

1. **Start from business impact (RTO/RPO)**, not from products.
2. **Immutable backup** is the single most important ransomware control — an attacker with stolen admin creds must not be able to delete your recovery point.
3. **Identity protections (MFA + PIM) come first** because most incidents start with a stolen credential.
4. **Untested backups are no backups.** Plan a monthly restore drill.
5. Phasing on the exam: **identity → backup → detect → segment → harden endpoints**.
6. **Break-glass accounts are protected, not unprotected.** Phishing-resistant MFA + excluded from blocking Conditional Access policies. "No MFA" is retired advice.

> ⚠️ Two different things share the abbreviation **ASR** in Microsoft security. *Azure Site Recovery* = replication/failover for BCDR. *Attack surface reduction rules* = Defender for Endpoint hardening rules (block Office macros spawning children, block credential theft from LSASS, …). Read the context before you answer.

**Next:** [Notebook 2 — Frameworks and Zero Trust](02_frameworks_and_zero_trust.ipynb)